In [ ]:
# !pip install --upgrade langchain
# !pip install langchain_community
# !pip install langchain_text_splitter
# !pip install langchain_chroma
# !pip install langchain_groq
# !pip install langchain

In [ ]:
import os
#from langchain.document_loaders import UnstructuredFileLoader #old import not working
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

/tmp/ipykernel_6062/1170151002.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredFileLoader


In [ ]:
os.environ["GROQ_API_KEY"] = "PASTE API KEY"

In [ ]:
import requests
url = "https://dspmuranchi.ac.in/pdf/Blog/Python%20Built-In%20Functions.pdf"
response = requests.get(url)

In [ ]:
with open("python_inbuildfunction.pdf", "wb") as f:
  f.write(response.content)

In [ ]:
# ! pip install unstructured

In [ ]:
loader = UnstructuredFileLoader("python_inbuildfunction.pdf")

/tmp/ipykernel_6062/2113599961.py:1: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader("python_inbuildfunction.pdf")


In [ ]:
loader

In [ ]:
documents = loader.load()
documents

In [ ]:
# !pip install "unstructured[pdf]"

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap=400
)

In [ ]:
texts = text_splitter.split_documents(documents)

In [ ]:
type(texts)

list

In [ ]:
len(texts)

7

In [ ]:
embeddings = HuggingFaceEmbeddings()

In [ ]:
persist_directory = "vector_db"

In [ ]:
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [ ]:
retriever = vectordb.as_retriever()

In [ ]:
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0
)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [ ]:
query = "what are the function from this pdf "
response = qa_chain.invoke({"query": query})

In [ ]:
print(response)

{'query': 'what are the function from this pdf ', 'result': 'Based on the provided text, the following Python built-in functions are listed:\n\n1.  `divmod()`\n2.  `enumerate()`\n3.  `eval()`\n4.  `exec()`\n5.  `filter()`\n6.  `float()`\n7.  `format()`\n8.  `frozenset()`\n9.  `getattr()`\n10. `globals()`\n11. `input()`\n12. `int()`\n13. `isinstance()`\n14. `ord()`\n15. `pow()`\n16. `print()`\n17. `range()`\n18. `repr()`\n19. `reversed()`\n20. `round()`\n21. `abs()`\n22. `all()`\n23. `any()`\n24. `ascii()`\n25. `bin()`\n26. `bool()`\n27. `bytearray()`\n28. `bytes()`\n29. `set()`\n30. `setattr()`', 'source_documents': [Document(id='75bef4fe-c658-46ad-b294-e16e8c9ff3c3', metadata={'source': 'python_inbuildfunction.pdf'}, page_content='>>> divmod(3,7)\n\n6 | P a g e\n\n(0, 3)\n\n>>> divmod(7,3)\n\n(2, 1) If you encounter any doubt in Python Built-in Function, Please Comment.\n\n17. enumerate()\n\nThis Python Built In function returns an enumerate object. In other words, it adds a counter t

In [ ]:
print(response["result"])

Based on the provided text, the following Python built-in functions are listed:

1.  `divmod()`
2.  `enumerate()`
3.  `eval()`
4.  `exec()`
5.  `filter()`
6.  `float()`
7.  `format()`
8.  `frozenset()`
9.  `getattr()`
10. `globals()`
11. `input()`
12. `int()`
13. `isinstance()`
14. `ord()`
15. `pow()`
16. `print()`
17. `range()`
18. `repr()`
19. `reversed()`
20. `round()`
21. `abs()`
22. `all()`
23. `any()`
24. `ascii()`
25. `bin()`
26. `bool()`
27. `bytearray()`
28. `bytes()`
29. `set()`
30. `setattr()`


In [ ]:
print(response["source_documents"][0].metadata["source"])

python_inbuildfunction.pdf


In [ ]:
import gradio as gr

def ask_pdf(question):
    response = qa_chain.invoke({"query": question})

    answer = response["result"]

    sources = ""
    if "source_documents" in response:
        sources = "\n\n".join([
            f"Source: {doc.metadata.get('source', 'Unknown')}, "
            f"Page: {doc.metadata.get('page', 'Unknown')}"
            for doc in response["source_documents"]
        ])

    return answer, sources


demo = gr.Interface(
    fn=ask_pdf,
    inputs=gr.Textbox(
        label="Ask a question about your PDF",
        placeholder="What are the functions mentioned in this PDF?"
    ),
    outputs=[
        gr.Textbox(label="Answer"),
        gr.Textbox(label="Sources")
    ],
    title="PDF Question Answering",
    description="Ask questions about the uploaded PDF."
)

demo.launch()